# Delta Ops — Small Files & OPTIMIZE

Copy `globalmart.gold.fact_sales` into **`fact_sales_fragmented`** with 100 Spark partitions (many small files), then run **OPTIMIZE**. Compare **DESCRIBE DETAIL** before and after.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.delta_ops.small_files as small_files_module
importlib.reload(small_files_module)

from src.delta_ops.small_files import SmallFilesConfig, run_small_files_optimize
from src.delta_ops.table_stats import describe_detail_summary
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = SmallFilesConfig(num_partitions=100)
print("Loaded from:", small_files_module.__file__)

In [ ]:
report = run_small_files_optimize(spark, config)
print("Before:", report["describe_detail_before"])
print("After:", report["describe_detail_after"])
display(spark.sql(f"DESCRIBE DETAIL {config.target_table}"))

In [ ]:
import json

print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/delta_small_files.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)